# Phase 1 — Training Hyperparameter Exploration

Reproduces **Methods 2.4.1** and **Results 3.1** of *Automated Segmentation and Morphometric Assessment of Neuronal Somata in the Rat Trigeminal Ganglion Using a Fine-Tuned Cellpose-SAM Model*.

Cellpose-SAM is fine-tuned on a fixed hold-out split (21 train / 5 test) with `batch_size = 1` and no data augmentation. Three hyperparameters are explored sequentially, each fixed at its selected value before the next one is varied:

| Sub-phase | Variable | Search space | Fixed |
|-----------|----------|--------------|-------|
| 1A | `n_epochs` | 1, 10, 20, 40, 60, 80, 100, 120, 140, 160, 180, 200, 220, 260, 300, 350, 400, 500 | `lr = 1e-5`, `wd = 0.1` |
| 1B | `learning_rate` | 1e-6, 5e-6, 1e-5, 2e-5, 5e-5, 1e-4, 1e-3 | `n_epochs` from 1A, `wd = 0.1` |
| 1C | `weight_decay` | 1e-4, 1e-3, 1e-2, 5e-2, 0.1, 0.5 | `n_epochs` from 1A, `lr` from 1B |
| 1D | `lr × wd` grid | immediate neighbours of the selected values | `n_epochs` from 1A |

Selection metric: AP@50, with the one-standard-error rule (most parsimonious configuration within one SE of the best). Inference: `diameter = None`, `channel_axis = -1`, other parameters at Cellpose defaults.

**Notebook series** (run in this order; each notebook reads the outputs of the previous ones):

| # | Notebook | Manuscript |
|---|----------|------------|
| 1 | `Phase1_training_hyperparameter_exploration.ipynb` | Methods 2.4.1 · Results 3.1 |
| 2 | `Phase2_generalization_data_augmentation.ipynb` | Methods 2.4.2 · Results 3.2 |
| 3 | `Phase3_inference_parameter_optimization.ipynb` | Methods 2.4.3 · Results 3.3 |
| 4 | `Phase4_morphology_measurements.ipynb` | Methods 2.5 · Results 3.4 · Table 1 · Figures 8–9 |
| 5 | `Phase5_figures.ipynb` | Figures 2–7 (no GPU needed) |

## Required files

| Path variable | Content | Provided file |
|---------------|---------|---------------|
| `TRAIN_DIR` | 21 training images (`<id>.tif`) and their annotations (`<id>_seg.npy`) | `dataset/train/` |
| `TEST_DIR` | 5 hold-out test images (`1`, `7`, `10`, `15`, `17`) and their annotations | `dataset/test/` |

## Outputs (`RESULTS_DIR = PROJECT_DIR/results/phase1`)

| File | Content |
|------|---------|
| `dataset_manifest.json` | Train/test image names of the fixed split |
| `<sub-phase>/models/*.pth` | One checkpoint per trained configuration |
| `<sub-phase>/metrics/*_metrics.json` | Aggregate and per-image metrics, training time, loss curve |
| `<sub-phase>/per_image_metrics/*_per_image.csv` | Per-image metrics |
| `<sub-phase>/results_final.json` | All runs of the sub-phase (input of `Phase5_figures`) |
| `<sub-phase>/<sub-phase>_summary.csv` | One row per run |
| `phase1_all_results_consolidated.csv` | All sub-phases (input of `Phase5_figures`) |
| `phase1_final_summary.json` | Selected configuration and search spaces |

**Expected folder layout** (files are distributed separately from the notebooks; set `PROJECT_DIR` in the *Paths* cell to the folder that contains them):

```
PROJECT_DIR/
├── dataset/
│   ├── train/        21 images <id>.tif + Cellpose annotations <id>_seg.npy
│   ├── test/          5 images (1, 7, 10, 15, 17) + <id>_seg.npy
│   └── validation/    5 images (3.2.2, Valid1–Valid4) + <id>_seg.npy
├── models/
│   ├── A_no_augmentation/     A_no_augmentation_fold_{1..5}_ep100_lr1e-05_wd0.1.pth
│   └── B_with_augmentation/   B_with_augmentation_fold_{1..5}_ep100_lr1e-05_wd0.1.pth
├── results/
│   ├── phase1/  phase2/  phase3/  phase4/
└── figures/
```

**Environment used in the study:** Google Colab Pro, NVIDIA RTX PRO 6000 Blackwell Server Edition GPU, Python 3, `cellpose==4.2.1.1` (Cellpose-SAM), PyTorch 2.11 (CUDA 12.8). Fixed seed: `RANDOM_SEED = 42`.

## 1. Environment

In [ ]:
!pip install --quiet "cellpose==4.2.1.1" albumentations

In [ ]:
import gc
import json
import time
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import torch

import cellpose
from cellpose import io, models, train
from cellpose.metrics import average_precision

print(f"cellpose {cellpose.version} | torch {torch.__version__} | numpy {np.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Paths

Only this cell needs to be edited. On Google Colab, mount Google Drive first (`from google.colab import drive; drive.mount('/content/drive')`) and point `PROJECT_DIR` to the folder that contains `dataset/`, `models/` and `results/`.

In [ ]:
PROJECT_DIR = Path(".")

TRAIN_DIR = PROJECT_DIR / "dataset" / "train"
TEST_DIR = PROJECT_DIR / "dataset" / "test"
RESULTS_DIR = PROJECT_DIR / "results" / "phase1"

## 3. Parameters (fixed by the study design)

In [ ]:
EPOCHS_TO_TEST = [1, 10, 20, 40, 60, 80, 100, 120, 140, 160, 180, 200, 220, 260, 300, 350, 400, 500]
LEARNING_RATES_TO_TEST = [1e-6, 5e-6, 1e-5, 2e-5, 5e-5, 1e-4, 1e-3]
WEIGHT_DECAYS_TO_TEST = [1e-4, 1e-3, 1e-2, 5e-2, 1e-1, 5e-1]

DEFAULT_LR = 1e-5
DEFAULT_WD = 0.1
BATCH_SIZE = 1
IOU_THRESHOLDS = [0.5, 0.75, 0.9]

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

SUBPHASE_DIRS = {
    "1A_epochs": RESULTS_DIR / "1A_epochs",
    "1B_learning_rate": RESULTS_DIR / "1B_learning_rate",
    "1C_weight_decay": RESULTS_DIR / "1C_weight_decay",
    "1D_lr_wd_grid": RESULTS_DIR / "1D_lr_wd_grid",
}
for d in SUBPHASE_DIRS.values():
    for sub in ("models", "metrics", "per_image_metrics"):
        (d / sub).mkdir(parents=True, exist_ok=True)

## 4. Data loading

In [ ]:
def load_image_and_mask(img_path, mask_path):
    img = io.imread(str(img_path))
    if img.ndim == 3 and img.shape[-1] == 4:
        img = img[:, :, :3]
    mask = io.imread(str(mask_path))
    return img, mask


def load_dataset(folder_path):
    folder = Path(folder_path)
    img_files = sorted([
        f for f in folder.glob("*")
        if f.suffix.lower() in [".tif", ".tiff", ".png", ".jpg"]
        and "_mask" not in f.stem
        and "_masks" not in f.stem
        and "_seg" not in f.stem
    ])

    images, masks, names = [], [], []
    for img_file in img_files:
        candidates = [
            folder / f"{img_file.stem}_masks.tif",
            folder / f"{img_file.stem}_masks.png",
            folder / f"{img_file.stem}_mask.tif",
            folder / f"{img_file.stem}_seg.npy",
        ]
        mask_path = next((p for p in candidates if p.exists()), None)
        if mask_path is None:
            print(f"WARNING: no mask for {img_file.name}")
            continue

        if mask_path.suffix == ".npy":
            mask = np.load(str(mask_path), allow_pickle=True).item()["masks"]
            img = io.imread(str(img_file))
            if img.ndim == 3 and img.shape[-1] == 4:
                img = img[:, :, :3]
        else:
            img, mask = load_image_and_mask(img_file, mask_path)

        images.append(img)
        masks.append(mask)
        names.append(img_file.stem)

    return images, masks, names


train_images, train_masks, train_names = load_dataset(TRAIN_DIR)
test_images, test_masks, test_names = load_dataset(TEST_DIR)

print(f"Train: {len(train_images)} images {train_names}")
print(f"Test:  {len(test_images)} images {test_names}")

In [ ]:
test_manifest = {
    "n_train": len(train_images),
    "n_test": len(test_images),
    "train_image_names": train_names,
    "test_image_names": test_names,
    "fixed_split": True,
    "random_seed": RANDOM_SEED,
    "ground_truth_neurons_test": int(sum(len(np.unique(m)) - 1 for m in test_masks)),
}
with open(RESULTS_DIR / "dataset_manifest.json", "w") as f:
    json.dump(test_manifest, f, indent=4)
print(json.dumps(test_manifest, indent=2))

## 5. Evaluation and training functions

Object-level metrics from `cellpose.metrics.average_precision` at IoU 0.50, 0.75 and 0.90: AP = TP / (TP + FP + FN), Precision = TP / (TP + FP), Recall = TP / (TP + FN), F1 = 2PR / (P + R). Aggregate AP is the mean of the per-image AP values.

In [ ]:
def compute_metrics_at_threshold(test_labels, predicted_masks, threshold):
    ap, tp, fp, fn = average_precision(test_labels, predicted_masks, threshold=[threshold])

    ap_per_image = ap.flatten().tolist()
    tp_per_image = tp.flatten().astype(int).tolist()
    fp_per_image = fp.flatten().astype(int).tolist()
    fn_per_image = fn.flatten().astype(int).tolist()

    total_tp, total_fp, total_fn = int(np.sum(tp)), int(np.sum(fp)), int(np.sum(fn))
    precision = total_tp / (total_tp + total_fp + 1e-8)
    recall = total_tp / (total_tp + total_fn + 1e-8)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-8)

    precision_per_image = [t / (t + f + 1e-8) for t, f in zip(tp_per_image, fp_per_image)]
    recall_per_image = [t / (t + f + 1e-8) for t, f in zip(tp_per_image, fn_per_image)]
    f1_per_image = [2 * (p * r) / (p + r + 1e-8) for p, r in zip(precision_per_image, recall_per_image)]

    return {
        "aggregate": {
            "AP": float(np.mean(ap)), "TP": total_tp, "FP": total_fp, "FN": total_fn,
            "Precision": float(precision), "Recall": float(recall), "F1": float(f1),
        },
        "per_image": {
            "AP": ap_per_image, "TP": tp_per_image, "FP": fp_per_image, "FN": fn_per_image,
            "Precision": precision_per_image, "Recall": recall_per_image, "F1": f1_per_image,
        },
    }


def evaluate_model(model, test_data, test_labels, test_names_list):
    predicted_masks, inference_times = [], []
    for img in test_data:
        t0 = time.time()
        masks_pred, _, _ = model.eval(img, diameter=None, channel_axis=-1)
        inference_times.append(time.time() - t0)
        predicted_masks.append(masks_pred)

    results = {
        "thresholds": {},
        "per_image_thresholds": {},
        "inference_time_s": inference_times,
        "test_image_names": test_names_list,
    }
    for thr in IOU_THRESHOLDS:
        metrics = compute_metrics_at_threshold(test_labels, predicted_masks, thr)
        results["thresholds"][f"AP@{thr}"] = metrics["aggregate"]
        results["per_image_thresholds"][f"AP@{thr}"] = metrics["per_image"]
    return results


def free_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
def train_and_evaluate(n_epochs, learning_rate, weight_decay, subphase_dir, run_label,
                       train_data, train_labels, test_data, test_labels, test_names_list):
    model_name = f"{run_label}_ep{n_epochs}_lr{learning_rate}_wd{weight_decay}_bs{BATCH_SIZE}"
    print(f"\n{'=' * 70}\nTRAINING: {model_name}\n{'=' * 70}")

    free_gpu_memory()
    base_model = models.CellposeModel(gpu=True)

    start = time.time()
    _, train_losses, _ = train.train_seg(
        base_model.net,
        train_data=train_data,
        train_labels=train_labels,
        test_data=None,
        test_labels=None,
        batch_size=BATCH_SIZE,
        n_epochs=n_epochs,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        nimg_per_epoch=len(train_data),
        model_name=model_name,
        save_every=max(n_epochs, 1),
    )
    train_time = time.time() - start

    final_model_path = subphase_dir / "models" / f"{model_name}.pth"
    base_model.net.save_model(str(final_model_path))
    print(f"Training time: {train_time:.2f}s")

    eval_start = time.time()
    eval_results = evaluate_model(base_model, test_data, test_labels, test_names_list)
    eval_time = time.time() - eval_start

    full_results = {
        "model_name": model_name,
        "model_file": final_model_path.name,
        "hyperparameters": {
            "n_epochs": n_epochs,
            "learning_rate": learning_rate,
            "weight_decay": weight_decay,
            "batch_size": BATCH_SIZE,
            "diameter": None,
            "channel_axis": -1,
        },
        "training_time_seconds": train_time,
        "evaluation_time_seconds": eval_time,
        "train_loss_final": float(train_losses[-1]) if len(train_losses) > 0 else None,
        "train_losses_curve": [float(x) for x in train_losses],
        "test_image_names": test_names_list,
        "aggregate_metrics": eval_results["thresholds"],
        "per_image_metrics": eval_results["per_image_thresholds"],
        "inference_time_per_image_s": eval_results["inference_time_s"],
    }
    with open(subphase_dir / "metrics" / f"{model_name}_metrics.json", "w") as f:
        json.dump(full_results, f, indent=4)

    records = []
    for i, img_name in enumerate(test_names_list):
        record = {"image_name": img_name, "model": model_name, "n_epochs": n_epochs,
                  "learning_rate": learning_rate, "weight_decay": weight_decay}
        for thr in IOU_THRESHOLDS:
            key = f"AP@{thr}"
            per_image = eval_results["per_image_thresholds"][key]
            for m in ("AP", "TP", "FP", "FN", "Precision", "Recall", "F1"):
                record[f"{key}_{m}"] = per_image[m][i]
        records.append(record)
    pd.DataFrame(records).to_csv(subphase_dir / "per_image_metrics" / f"{model_name}_per_image.csv", index=False)

    agg = eval_results["thresholds"]
    print(f"AP@0.5 = {agg['AP@0.5']['AP']:.4f} | AP@0.75 = {agg['AP@0.75']['AP']:.4f} | "
          f"AP@0.9 = {agg['AP@0.9']['AP']:.4f} | F1@0.5 = {agg['AP@0.5']['F1']:.4f}")

    del base_model
    free_gpu_memory()
    return full_results


def run_subphase(subphase, run_label, configs):
    subphase_dir = SUBPHASE_DIRS[subphase]
    results = []
    for n_epochs, lr, wd in configs:
        try:
            result = train_and_evaluate(n_epochs, lr, wd, subphase_dir, run_label,
                                        train_images, train_masks, test_images, test_masks, test_names)
        except Exception as e:
            print(f"ERROR at n_epochs={n_epochs}, lr={lr}, wd={wd}: {e}")
            free_gpu_memory()
            continue
        results.append(result)
        with open(subphase_dir / "results_partial.json", "w") as f:
            json.dump(results, f, indent=4)
        print(f"[{run_label} progress: {len(results)}/{len(configs)}]")
    with open(subphase_dir / "results_final.json", "w") as f:
        json.dump(results, f, indent=4)
    return consolidate_subphase_results(results, subphase_dir, subphase)


def consolidate_subphase_results(subphase_results, subphase_dir, subphase_name):
    records = []
    for r in subphase_results:
        record = {
            "n_epochs": r["hyperparameters"]["n_epochs"],
            "learning_rate": r["hyperparameters"]["learning_rate"],
            "weight_decay": r["hyperparameters"]["weight_decay"],
            "model_name": r["model_name"],
            "training_time_s": r["training_time_seconds"],
            "train_loss_final": r["train_loss_final"],
        }
        for thr in IOU_THRESHOLDS:
            key = f"AP@{thr}"
            for m in ("AP", "TP", "FP", "FN", "Precision", "Recall", "F1"):
                record[f"{key}_{m}"] = r["aggregate_metrics"][key][m]
        records.append(record)
    df = pd.DataFrame(records)
    df.to_csv(subphase_dir / f"{subphase_name}_summary.csv", index=False)
    return df


def one_standard_error(df, group_col, subphase):
    peak_idx = df["AP@0.5_AP"].idxmax()
    peak_ap = float(df.loc[peak_idx, "AP@0.5_AP"])
    per_image = pd.read_csv(SUBPHASE_DIRS[subphase] / "per_image_metrics" / f"{df.loc[peak_idx, 'model_name']}_per_image.csv")
    sd = float(per_image["AP@0.5_AP"].std(ddof=1))
    se = sd / np.sqrt(len(per_image))
    return {"peak_value": df.loc[peak_idx, group_col], "peak_ap": peak_ap,
            "sd": sd, "se": se, "threshold": peak_ap - se}

## 6. Sub-phase 1A — Number of epochs

`lr = 1e-5`, `wd = 0.1` (Cellpose defaults). The number of epochs is selected with the one-standard-error rule: the smallest number of epochs whose AP@50 is within one SE (per-image SD / √n) of the peak.

In [ ]:
df_1A = run_subphase("1A_epochs", "1A", [(ep, DEFAULT_LR, DEFAULT_WD) for ep in EPOCHS_TO_TEST])
df_1A = df_1A.sort_values("n_epochs").reset_index(drop=True)
df_1A[["n_epochs", "AP@0.5_AP", "AP@0.75_AP", "AP@0.9_AP", "AP@0.5_F1", "training_time_s"]]

In [ ]:
se_1A = one_standard_error(df_1A, "n_epochs", "1A_epochs")
within = df_1A[df_1A["AP@0.5_AP"] >= se_1A["threshold"]]
BEST_EPOCHS = int(within["n_epochs"].min())

print(f"Peak: {se_1A['peak_value']} epochs, AP@50 = {se_1A['peak_ap']:.3f} ± {se_1A['sd']:.3f} (SD)")
print(f"One-SE threshold: {se_1A['threshold']:.3f}")
print(f"Selected n_epochs (most parsimonious within 1 SE): {BEST_EPOCHS}")

## 7. Sub-phase 1B — Learning rate

`n_epochs` from 1A, `wd = 0.1`. The Cellpose-SAM recommended learning rate (1e-5) is adopted when it lies within one SE of the best learning rate.

In [ ]:
df_1B = run_subphase("1B_learning_rate", "1B", [(BEST_EPOCHS, lr, DEFAULT_WD) for lr in LEARNING_RATES_TO_TEST])
df_1B = df_1B.sort_values("learning_rate").reset_index(drop=True)
df_1B[["learning_rate", "AP@0.5_AP", "AP@0.75_AP", "AP@0.9_AP", "AP@0.5_F1"]]

In [ ]:
RECOMMENDED_LR = 1e-5

se_1B = one_standard_error(df_1B, "learning_rate", "1B_learning_rate")
rec_ap = float(df_1B.loc[np.isclose(df_1B["learning_rate"], RECOMMENDED_LR), "AP@0.5_AP"].iloc[0])
if rec_ap < se_1B["threshold"]:
    print("WARNING: the recommended learning rate is not within one SE of the peak.")
BEST_LR = RECOMMENDED_LR

print(f"Peak: lr = {se_1B['peak_value']:.0e}, AP@50 = {se_1B['peak_ap']:.3f}")
print(f"One-SE threshold: {se_1B['threshold']:.3f} | lr = {RECOMMENDED_LR:.0e}: AP@50 = {rec_ap:.3f}")
print(f"Selected learning_rate: {BEST_LR:.0e}")

## 8. Sub-phase 1C — Weight decay

`n_epochs` from 1A, `lr` from 1B. The Cellpose-SAM recommended weight decay (0.1) is adopted when it lies within one SE of the best weight decay.

In [ ]:
df_1C = run_subphase("1C_weight_decay", "1C", [(BEST_EPOCHS, BEST_LR, wd) for wd in WEIGHT_DECAYS_TO_TEST])
df_1C = df_1C.sort_values("weight_decay").reset_index(drop=True)
df_1C[["weight_decay", "AP@0.5_AP", "AP@0.75_AP", "AP@0.9_AP", "AP@0.5_F1"]]

In [ ]:
RECOMMENDED_WD = 0.1

se_1C = one_standard_error(df_1C, "weight_decay", "1C_weight_decay")
rec_ap = float(df_1C.loc[np.isclose(df_1C["weight_decay"], RECOMMENDED_WD), "AP@0.5_AP"].iloc[0])
if rec_ap < se_1C["threshold"]:
    print("WARNING: the recommended weight decay is not within one SE of the peak.")
BEST_WD = RECOMMENDED_WD

print(f"Peak: wd = {se_1C['peak_value']:g}, AP@50 = {se_1C['peak_ap']:.3f}")
print(f"One-SE threshold: {se_1C['threshold']:.3f} | wd = {RECOMMENDED_WD:g}: AP@50 = {rec_ap:.3f}")
print(f"Selected weight_decay: {BEST_WD:g}")

## 9. Sub-phase 1D — Learning rate × weight decay grid

3 × 3 grid formed by the selected values and their immediate neighbours in the search spaces, at `n_epochs` from 1A.

In [ ]:
def get_neighbors(value, all_values, n_neighbors=1):
    sorted_vals = sorted(set(all_values) | {value})
    idx = sorted_vals.index(value)
    return sorted_vals[max(0, idx - n_neighbors): idx + n_neighbors + 1]


REFINED_LRS = get_neighbors(BEST_LR, LEARNING_RATES_TO_TEST)
REFINED_WDS = get_neighbors(BEST_WD, WEIGHT_DECAYS_TO_TEST)
print(f"Learning rates: {REFINED_LRS}")
print(f"Weight decays:  {REFINED_WDS}")

df_1D = run_subphase("1D_lr_wd_grid", "1D",
                     [(BEST_EPOCHS, lr, wd) for lr, wd in product(REFINED_LRS, REFINED_WDS)])
df_1D.pivot(index="learning_rate", columns="weight_decay", values="AP@0.5_AP")

In [ ]:
best_idx_1D = df_1D["AP@0.5_AP"].idxmax()
FINAL_BEST_EPOCHS = int(df_1D.loc[best_idx_1D, "n_epochs"])
FINAL_BEST_LR = float(df_1D.loc[best_idx_1D, "learning_rate"])
FINAL_BEST_WD = float(df_1D.loc[best_idx_1D, "weight_decay"])

print(f"Final configuration: n_epochs = {FINAL_BEST_EPOCHS}, lr = {FINAL_BEST_LR:g}, wd = {FINAL_BEST_WD:g}")
for thr in ("0.5", "0.75", "0.9"):
    print(f"AP@{thr}: {df_1D.loc[best_idx_1D, f'AP@{thr}_AP']:.4f}")

## 10. Consolidation

In [ ]:
all_dfs = []
for name, df in (("1A_epochs", df_1A), ("1B_learning_rate", df_1B),
                 ("1C_weight_decay", df_1C), ("1D_lr_wd_grid", df_1D)):
    all_dfs.append(df.assign(subphase=name))
df_all_phase1 = pd.concat(all_dfs, ignore_index=True)
df_all_phase1.to_csv(RESULTS_DIR / "phase1_all_results_consolidated.csv", index=False)

final_summary = {
    "selected": {"n_epochs": BEST_EPOCHS, "learning_rate": BEST_LR, "weight_decay": BEST_WD},
    "grid_1D_best": {"n_epochs": FINAL_BEST_EPOCHS, "learning_rate": FINAL_BEST_LR,
                     "weight_decay": FINAL_BEST_WD,
                     "AP@0.5": float(df_1D.loc[best_idx_1D, "AP@0.5_AP"]),
                     "AP@0.75": float(df_1D.loc[best_idx_1D, "AP@0.75_AP"]),
                     "AP@0.9": float(df_1D.loc[best_idx_1D, "AP@0.9_AP"]),
                     "F1@0.5": float(df_1D.loc[best_idx_1D, "AP@0.5_F1"])},
    "one_standard_error": {"1A_epochs": se_1A, "1B_learning_rate": se_1B, "1C_weight_decay": se_1C},
    "search_spaces": {"epochs": EPOCHS_TO_TEST, "learning_rates": LEARNING_RATES_TO_TEST,
                      "weight_decays": WEIGHT_DECAYS_TO_TEST,
                      "refined_lr_grid_1D": REFINED_LRS, "refined_wd_grid_1D": REFINED_WDS},
    "total_models_trained": len(df_all_phase1),
    "dataset": test_manifest,
}
with open(RESULTS_DIR / "phase1_final_summary.json", "w") as f:
    json.dump(final_summary, f, indent=4, default=float)

print(f"Models trained: {len(df_all_phase1)}")
print(json.dumps(final_summary["grid_1D_best"], indent=2))